In [1]:
# Standard library
import os
import sys
import io
import logging
import time
import random
import re
from datetime import datetime

# Third-party libraries
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm
import bs4

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


In [2]:
# Windows UTF-8 encoding for console output
if sys.platform == 'win32' and hasattr(sys.stdout, 'buffer'):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

# Configure logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('scraping.log', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

print("✅ Logger configured successfully")

✅ Logger configured successfully


In [3]:
# Session with retries and polite backoff
def build_session():
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        ),
        "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    })
    return s

session = build_session()


def fetch(url: str, timeout: int = 15):
    """GET with retries + explicit 503 handling + jittered delay."""
    max_attempts = 6
    for attempt in range(max_attempts):
        try:
            resp = session.get(url, timeout=timeout)
            if resp.status_code == 503:
                sleep_time = min(60, 2 ** attempt + random.uniform(0.2, 0.8))
                logger.warning(f"503 Service Unavailable: backoff {sleep_time:.1f}s (attempt {attempt+1}/{max_attempts}) - {url}")
                time.sleep(sleep_time)
                continue
            resp.raise_for_status()
            # Polite delay with jitter after successful request
            time.sleep(0.5 + random.uniform(0.2, 0.6))
            return resp
        except requests.RequestException as e:
            sleep_time = min(60, 2 ** attempt + random.uniform(0.2, 0.8))
            logger.warning(f"Request error: {e}. Sleep {sleep_time:.1f}s (attempt {attempt+1}/{max_attempts}) - {url}")
            time.sleep(sleep_time)
    logger.error(f"Failed after retries: {url}")
    return None

print("✅ Session and fetch() configured successfully")


✅ Session and fetch() configured successfully


In [4]:
# Get the total number of pages
base_url = "https://www.assemblee-nationale.fr/dyn/17/scrutins"
first_response = fetch(f"{base_url}?order=date,desc&limit=100")
if first_response is None:
    raise RuntimeError("Unable to fetch the first page, even after retries.")
first_soup = bs4.BeautifulSoup(first_response.text, 'html.parser')

# Find all pagination items and get the max page number
pagination_items = first_soup.find_all('div', class_='an-pagination--item')
page_numbers = []

for item in pagination_items:
    span = item.find('span')
    if span and span.get_text(strip=True).isdigit():
        page_numbers.append(int(span.get_text(strip=True)))

max_page = max(page_numbers) if page_numbers else 1
print(f"Total pages found: {max_page}")

# Now scrape all pages
all_scrutins = []

for page in tqdm(range(1, max_page + 1)):
    print(f"\nScraping page {page}/{max_page}...")
    
    url = f"{base_url}?order=date,desc&limit=100&page={page}"
    
    try:
        response = fetch(url)
        if response is None:
            logger.error(f"Skip page {page}: could not fetch after retries")
            continue
        soup = bs4.BeautifulSoup(response.text, 'html.parser')
        
        # Find all scrutin blocks
        scrutin_blocks = soup.find_all('div', class_='an-bloc _style-scrutin')
        
        if not scrutin_blocks:
            print(f"No scrutins found on page {page}. Stopping.")
            break
        
        # Extract links from each block
        for block in scrutin_blocks:
            link = block.find('a', class_='link h6')
            if link:
                href = link.get('href')
                scrutin_titre = link.get_text(strip=True)
                scrutin_num = href.split('/')[-1] if href else None
                
                all_scrutins.append({
                    'scrutin_url': f"https://www.assemblee-nationale.fr{href}" if href else None,
                    'scrutin_numero': scrutin_num,
                    'scrutin_titre': scrutin_titre
                })
        
        print(f"  Found {len(scrutin_blocks)} scrutins on page {page}")
        
    except Exception as e:
        print(f"Error on page {page}: {e}")
        continue

print(f"\nTotal scrutins scraped: {len(all_scrutins)}")

# Create DataFrame and remove duplicates
df_scrutins = pd.DataFrame(all_scrutins)
df_scrutins = df_scrutins.drop_duplicates(subset=['scrutin_numero'])
print(f"Unique scrutins: {len(df_scrutins)}")
print(f"\nFirst 3 scrutins:")
for idx, row in df_scrutins.head(3).iterrows():
    print(f"  {row['scrutin_numero']}: {row['scrutin_titre']}")


Total pages found: 50


  0%|          | 0/50 [00:00<?, ?it/s]


Scraping page 1/50...


  2%|▏         | 1/50 [00:02<02:13,  2.72s/it]

  Found 100 scrutins on page 1

Scraping page 2/50...


...

  Found 100 scrutins on page 48

Scraping page 49/50...


 98%|█████████▊| 49/50 [02:29<00:04,  4.79s/it]

  Found 100 scrutins on page 49

Scraping page 50/50...


100%|██████████| 50/50 [02:30<00:00,  3.02s/it]

  Found 71 scrutins on page 50

Total scrutins scraped: 4948
Unique scrutins: 4947

First 3 scrutins:
  4942: Scrutin public n°4942 sur l'article premier du projet de loi spéciale prévue par l'article 45 de la loi organique du 1er août 2001 relative aux lois de finances (première lecture).
  4943: Scrutin public n°4943 sur l'amendement n° 5 de M. Jean-Philippe Tanguy après l'article premier du projet de loi spéciale prévue par l'article 45 de la loi organique du 1er août 2001 relative aux lois de finances (première lecture).
  4944: Scrutin public n°4944 sur l'amendement n° 17 de la commission des finances et l'amendement identique suivant à l'article 2 du projet de loi spéciale prévue par l'article 45 de la loi organique du 1er août 2001 relative aux lois de finances (première lecture).


In [5]:
# Save scrutin links to CSV
output_dir = os.path.dirname(os.getcwd()) + "/0_raw"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

csv_path = f"{output_dir}/scrutins_links.csv"
df_scrutins.to_csv(csv_path, index=False)
logger.info(f"✅ Saved {len(df_scrutins)} scrutin links to {csv_path}")
print(f"✅ Scrutin links saved: {csv_path}")

2026-01-06 10:26:35,611 - INFO - ✅ Saved 4947 scrutin links to c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\17e_legislature_18juillet2024/0_raw/scrutins_links.csv
✅ Scrutin links saved: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\17e_legislature_18juillet2024/0_raw/scrutins_links.csv


## Target Schema for Each Vote Record

Ideally we want these features for each voter of each scrutiny:

```python
['votant/acteurRef',
 'votant/mandatRef',
 'votant/causePositionVote',
 'groupe/organeRef',
 'groupe/nombreMembresGroupe',
 'groupe/vote/positionMajoritaire',
 'groupe/vote/decompteVoix/nonVotants',
 'groupe/vote/decompteVoix/pour',
 'groupe/vote/decompteVoix/contre',
 'groupe/vote/decompteVoix/abstentions',
 'groupe/vote/decompteVoix/nonVotantsVolontaires',
 'main/uid',
 'main/numero',
 'main/organeRef',
 'main/legislature',
 'main/seanceRef',
 'main/dateScrutin',
 'main/quantiemeJourSeance',
 'main/typeVote/codeTypeVote',
 'main/typeVote/typeMajorité',
 'main/sort/code',
 'main/titre',
 'main/objet/libelle',
 'main/syntheseVote/nombreVotants',
 'main/syntheseVote/suffragesExprimes',
 'main/syntheseVote/nbrSuffragesRequis',
 'main/syntheseVote/annonce',
 'main/syntheseVote/decompte/nonVotants',
 'main/syntheseVote/decompte/pour',
 'main/syntheseVote/decompte/contre',
 'main/syntheseVote/decompte/abstentions',
 'main/syntheseVote/decompte/nonVotantsVolontaires',
 'votant/vote',
 'votant/parDelegation']
 ```


In [17]:
def scrape_scrutin_details(scrutin_url, scrutin_numero):
    """Scrape voting details from a single scrutin page."""
    vote_records = []
    
    try:
        resp = fetch(scrutin_url)
        if resp is None:
            logger.error(f"Could not fetch scrutin {scrutin_numero}")
            return vote_records
        
        soup = bs4.BeautifulSoup(resp.text, 'html.parser')
        
        # Initialize scrutin data structure
        scrutin_data = {
            'main/uid': scrutin_numero,
            'main/numero': scrutin_numero,
            'main/organeRef': None,
            'main/legislature': '17',
            'main/seanceRef': None,
            'main/dateScrutin': None,
            'main/quantiemeJourSeance': None,
            'main/typeVote/codeTypeVote': None,
            'main/typeVote/typeMajorité': None,
            'main/sort/code': None,
            'main/titre': None,
            'main/objet/libelle': None,
            'main/syntheseVote/nombreVotants': None,
            'main/syntheseVote/suffragesExprimes': None,
            'main/syntheseVote/nbrSuffragesRequis': None,
            'main/syntheseVote/annonce': None,
            'main/syntheseVote/decompte/nonVotants': None,
            'main/syntheseVote/decompte/pour': None,
            'main/syntheseVote/decompte/contre': None,
            'main/syntheseVote/decompte/abstentions': None,
            'main/syntheseVote/decompte/nonVotantsVolontaires': None,
        }
        
        # Extract date from h2.h4 (format: "Unique séance du [JOUR] [DATE] [MOIS] [ANNÉE]")
        date_header = soup.find('h2', class_='h4')
        if date_header:
            date_text = date_header.get_text(strip=True)
            try:
                match = re.search(r'(\d{1,2})\s+(\w+)\s+(\d{4})', date_text)
                if match:
                    day, month_fr, year = match.groups()
                    months_fr = {
                        'janvier': 1, 'février': 2, 'mars': 3, 'avril': 4,
                        'mai': 5, 'juin': 6, 'juillet': 7, 'août': 8,
                        'septembre': 9, 'octobre': 10, 'novembre': 11, 'décembre': 12
                    }
                    month_num = months_fr.get(month_fr.lower(), None)
                    if month_num:
                        date_obj = datetime(int(year), month_num, int(day))
                        scrutin_data['main/dateScrutin'] = date_obj.strftime('%Y-%m-%d')
            except Exception as e:
                logger.warning(f"Could not parse date for scrutin {scrutin_numero}: {e}")
        
        # Extract title from p.h6
        title_elem = soup.find('p', class_='h6')
        if title_elem:
            title_text = title_elem.get_text(strip=True)
            if 'Scrutin public' in title_text:
                scrutin_data['main/titre'] = title_text
        
        # Extract vote result announcement
        result_elem = soup.find('span', class_='_colored-green')
        if result_elem:
            result_text = result_elem.get_text(strip=True)
            scrutin_data['main/syntheseVote/annonce'] = result_text
        else:
            for span in soup.find_all('span', class_='_colored'):
                if 'Assemblée nationale' in span.get_text():
                    scrutin_data['main/syntheseVote/annonce'] = span.get_text(strip=True)
                    break
        
        # Generate sort/code from announcement
        if scrutin_data['main/syntheseVote/annonce']:
            if 'a adopté' in scrutin_data['main/syntheseVote/annonce'].lower():
                scrutin_data['main/sort/code'] = 'adopté'
            else:
                scrutin_data['main/sort/code'] = 'rejeté'
        
        # Extract vote synthesis
        vote_summary = soup.find_all('span', class_='_colored-primary')
        for elem in vote_summary:
            text = elem.get_text(strip=True)
            if 'Nombre de votants' in text:
                scrutin_data['main/syntheseVote/nombreVotants'] = text.split(':')[-1].strip()
            elif 'Nombre de suffrages exprimés' in text:
                scrutin_data['main/syntheseVote/suffragesExprimes'] = text.split(':')[-1].strip()
            elif 'Majorité absolue' in text:
                scrutin_data['main/syntheseVote/nbrSuffragesRequis'] = text.split(':')[-1].strip()
        
        # Extract group-based voting - ITERATE OVER ALL GROUPS
        groups_sections = soup.find_all('ul', class_='simple-tree-list')
        for groups_section in groups_sections:
            group_items = groups_section.find_all('li', recursive=False)
            
            for group_item in group_items:
                group_header = group_item.find('h3')
                if not group_header:
                    continue
                
                groupe_organeRef = group_item.get('data-organe-id', None)
                group_text = group_item.get_text(strip=True)
                member_match = re.search(r'\((\d+)\s*membres?\)', group_text)
                groupe_nombreMembresGroupe = member_match.group(1) if member_match else None
                
                # Extract individual deputy votes organized by vote type
                # Find the main <ul> inside the group (it contains vote blocks)
                deputy_list = group_item.find('ul')
                if deputy_list:
                    # Find all vote category blocks (Pour, Contre, Abstention, Non-votant)
                    vote_blocks = deputy_list.find_all('li', class_='relative-flex', recursive=False)
                    
                    for vote_block in vote_blocks:
                        # Extract vote type from the header
                        vote_header = vote_block.find('div', class_='_align-self-start')
                        votant_vote = None
                        if vote_header:
                            vote_label = vote_header.find('span', class_='h6')
                            if vote_label:
                                vote_text = vote_label.get_text(strip=True).lower()
                                if 'pour' in vote_text:
                                    votant_vote = 'pour'
                                elif 'contre' in vote_text:
                                    votant_vote = 'contre'
                                elif 'abstention' in vote_text:
                                    votant_vote = 'abstention'
                                elif 'non-votant' in vote_text or 'non votant' in vote_text:
                                    votant_vote = 'nonVotant'
                        
                        # Find all deputies in this vote category
                        deputies_ul = vote_block.find('ul', class_='_2-columns')
                        if deputies_ul and votant_vote:
                            deputies = deputies_ul.find_all('li', class_='_no-border', recursive=False)
                            
                            for deputy_item in deputies:
                                # Extract acteur and mandat refs from data-acteur-id attribute
                                acteur_data = deputy_item.get('data-acteur-id', '')
                                votant_acteurRef = None
                                votant_mandatRef = None
                                
                                if acteur_data and ' - ' in acteur_data:
                                    parts = acteur_data.split(' - ')
                                    votant_acteurRef = parts[0].strip()
                                    votant_mandatRef = parts[1].strip() if len(parts) > 1 else None
                                
                                record = {**scrutin_data}
                                record['votant/acteurRef'] = votant_acteurRef
                                record['votant/mandatRef'] = votant_mandatRef
                                record['votant/vote'] = votant_vote
                                record['votant/parDelegation'] = None
                                record['votant/causePositionVote'] = None
                                record['groupe/organeRef'] = groupe_organeRef
                                record['groupe/nombreMembresGroupe'] = groupe_nombreMembresGroupe
                                record['groupe/vote/positionMajoritaire'] = None
                                record['groupe/vote/decompteVoix/nonVotants'] = None
                                record['groupe/vote/decompteVoix/pour'] = None
                                record['groupe/vote/decompteVoix/contre'] = None
                                record['groupe/vote/decompteVoix/abstentions'] = None
                                record['groupe/vote/decompteVoix/nonVotantsVolontaires'] = None
                                
                                vote_records.append(record)
        
        if not vote_records:
            logger.warning(f"No votes extracted for scrutin {scrutin_numero}")
        
    except Exception as e:
        logger.error(f"Error scraping scrutin {scrutin_numero}: {e}")
    
    return vote_records

print("✅ scrape_scrutin_details() function defined")

✅ scrape_scrutin_details() function defined


In [20]:
# Main scraping loop with checkpointing
# Define output directory
output_dir = os.path.dirname(os.getcwd()) + "/1_extract_python"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

checkpoint_file = f"{output_dir}/scrutins_checkpoint.csv"
final_file = f"{output_dir}/scrutins.csv"

# Load already scraped scrutins
if os.path.exists(checkpoint_file):
    df_existing = pd.read_csv(checkpoint_file)
    already_scraped = set(df_existing['main/uid'].unique())
    all_vote_records = df_existing.to_dict('records')
    logger.info(f"Resuming: {len(already_scraped)} scrutins already processed")
else:
    already_scraped = set()
    all_vote_records = []
    logger.info("Starting fresh scraping")

# Load scrutin links
links_file = os.path.dirname(os.getcwd()) + "/0_raw/scrutins_links.csv"
df_scrutins_links = pd.read_csv(links_file)

# Filter scrutins not yet scraped
df_to_scrape = df_scrutins_links[~df_scrutins_links['scrutin_numero'].isin(already_scraped)]
logger.info(f"Scrutins to process: {len(df_to_scrape)}/{len(df_scrutins_links)}")

counter = 0
save_interval = 200

for idx, row in df_to_scrape.iterrows():
    scrutin_url = row['scrutin_url']
    scrutin_numero = row['scrutin_numero']
    
    votes = scrape_scrutin_details(scrutin_url, scrutin_numero)
    all_vote_records.extend(votes)
    counter += 1
    
    # Log every 50 iterations
    if counter % 50 == 0:
        logger.info(f"Progress: {counter}/{len(df_to_scrape)} scrutins processed")
    
    # Save checkpoint every 200 scrutins
    if counter % save_interval == 0:
        df_temp = pd.DataFrame(all_vote_records)
        df_temp.to_csv(checkpoint_file, index=False)
        logger.info(f"[CHECKPOINT] Saved: {len(all_vote_records)} votes ({counter} scrutins)")
    
    time.sleep(0.1)

# Create final DataFrame
df_scrutins_17 = pd.DataFrame(all_vote_records)

# Convert numeric columns to Int64 (nullable integer)
numeric_columns = df_scrutins_17.select_dtypes(include=['float64', 'float32']).columns.tolist()

if numeric_columns:
    logger.info(f"Converting {len(numeric_columns)} float columns to Int64")
    for col in numeric_columns:
        df_scrutins_17[col] = df_scrutins_17[col].astype('Int64')

# Save final dataset
df_scrutins_17.to_csv(final_file, index=False)
logger.info(f"[OK] Scraping complete: {len(df_scrutins_17)} votes saved to {final_file}")

# Clean up checkpoint
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)
    logger.info("Checkpoint removed")

# Summary
print(f"\n✅ SCRAPING COMPLETE")
print(f"  Total votes: {df_scrutins_17.shape[0]:,}")
print(f"  Unique députés: {df_scrutins_17['votant/acteurRef'].nunique()}")
print(f"  Unique scrutins: {df_scrutins_17['main/uid'].nunique()}")
print(f"  File saved: {final_file}")


2026-01-06 15:40:29,322 - INFO - Resuming: 1800 scrutins already processed
2026-01-06 15:40:29,399 - INFO - Scrutins to process: 3147/4947
2026-01-06 15:42:30,856 - INFO - Progress: 50/3147 scrutins processed
2026-01-06 15:44:25,518 - INFO - Progress: 100/3147 scrutins processed
2026-01-06 15:46:15,324 - INFO - Progress: 150/3147 scrutins processed
2026-01-06 15:48:08,421 - INFO - Progress: 200/3147 scrutins processed
2026-01-06 15:48:25,528 - INFO - [CHECKPOINT] Saved: 431018 votes (200 scrutins)
2026-01-06 15:50:13,778 - INFO - Progress: 250/3147 scrutins processed
2026-01-06 15:51:58,916 - INFO - Progress: 300/3147 scrutins processed
2026-01-06 15:53:45,847 - INFO - Progress: 350/3147 scrutins processed
2026-01-06 15:55:30,290 - INFO - Progress: 400/3147 scrutins processed
2026-01-06 15:55:48,651 - INFO - [CHECKPOINT] Saved: 445979 votes (400 scrutins)
2026-01-06 15:57:43,607 - INFO - Progress: 450/3147 scrutins processed
2026-01-06 15:59:31,821 - INFO - Progress: 500/3147 scrutins 